# 🧠 Reasoning Pruning: Interactive Exploration on Real Reasoning Datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avrymi-asraf/reasoning-pruning-agy/blob/master/notebooks/01_explore_pruning.ipynb)

This notebook provides an **end-to-end, live interactive laboratory** for exploring reasoning traces generated on authentic benchmark datasets, auditing them for skippable overthinking spans using Decision model $D$, visually rendering trace diffs, and extracting $(x \to y)$ transition datasets for 4-bit QLoRA fine-tuning.

### 🎯 The Core Pipeline:
$$\text{Real Benchmark Question } q \xrightarrow[\text{generate\_trace}]{\text{Generator } G} \text{Trace } (s_1..s_n) \xrightarrow[\text{find\_first\_skip}]{\text{Decision } D} \text{Skip } s_k \xrightarrow[\text{extract\_transition}]{\text{Pair}} (x \to y) \xrightarrow[\text{rollout\_pruning}]{\text{Recursive Rollout}} \text{PT Dataset}$$

| Live Tool | Purpose | Output Structure |
|---|---|---|
| `rp.load_spectrum_benchmarks` | Streams real questions across multi-domain reasoning benchmarks | `List[Dict[str, Any]]` |
| `rp.generate_trace` | Prompts generator $G$ and segments reasoning steps | `ReasoningTrace` |
| `rp.find_first_skip` | Audits trace live with decision model $D$ | `PruneDecision` |
| `rp.render_trace_diff` | Renders color-coded HTML diff of pruned steps | `HTML` / `RichPanel` |
| `rp.extract_transition` | Isolates $(x \to y)$ next-step training pair | `TransitionExample` |
| `rp.rollout_pruning` | Multi-depth iterative pruning and continuation | `RolloutResult` |
| `rp.build_pt_dataset` | Assembles Hugging Face Dataset across real questions | `datasets.Dataset` |

## 1. Setup, Environment & API Keys

Detects Google Colab to automatically clone the repo and install dependencies in editable mode (`pip install -e .`). Configure your LiteLLM model endpoints and API keys below.

In [ ]:
# 1. Environment bootstrap (Google Colab vs local workspace)
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("🚀 Running on Google Colab. Setting up environment and repository...")
    REPO_DIR = "/content/reasoning-pruning-agy"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/avrymi-asraf/reasoning-pruning-agy.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e .
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("✅ reasoning-pruning repository cloned and installed!")
else:
    print("💻 Running in local workspace.")

import json
import re
import pandas as pd
from IPython.display import HTML, display

# Import core reasoning_pruning tools
import reasoning_pruning as rp
from reasoning_pruning.types import (
    ReasoningTrace,
    PruneDecision,
    TransitionExample,
    RolloutResult,
)

# Configure API Keys for LiteLLM providers:
# os.environ["GEMINI_API_KEY"] = "your-gemini-key"
# os.environ["OPENAI_API_KEY"] = "your-openai-key"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-key"
# os.environ["DEEPSEEK_API_KEY"] = "your-deepseek-key"
# os.environ["HF_TOKEN"] = "your-hf-token"

print(f"✅ reasoning_pruning v{rp.__version__} loaded successfully.")

## 2. Load Real Reasoning Benchmark Spectrum

We load authentic benchmark questions across **6 distinct cognitive reasoning families** using `rp.load_spectrum_benchmarks`:
1. **Arithmetic & Word Math**: GSM8K (`openai/gsm8k`) & SVAMP (`ChilleD/SVAMP`)
2. **Scientific Reasoning**: ARC-Challenge (`allenai/ai2_arc`)
3. **Commonsense & Physical Logic**: CommonsenseQA (`tau/commonsense_qa`)
4. **Multi-hop & Deductive Logic**: HotpotQA (`hotpotqa/hotpot_qa`)
5. **Extractive & Span QA**: SQuAD 2.0 (`rajpurkar/squad_v2`)

In [ ]:
# Configure Generator G and Decision Auditor D
# Examples: "gemini/gemini-2.5-flash", "gpt-4o-mini", "claude-3-5-haiku-20241022", "deepseek/deepseek-chat"
MODEL_G = os.environ.get("RP_MODEL_G", "gemini/gemini-2.5-flash" if "GEMINI_API_KEY" in os.environ else "gpt-4o-mini")
MODEL_D = os.environ.get("RP_MODEL_D", "gemini/gemini-2.5-flash" if "GEMINI_API_KEY" in os.environ else "gpt-4o-mini")

print(f"Generator Model (G): {MODEL_G}")
print(f"Decision Model  (D): {MODEL_D}")

# Stream real questions across the multi-domain benchmark spectrum
print("\n📥 Loading real benchmark spectrum questions...")
REAL_SPECTRUM = rp.load_spectrum_benchmarks(
    benchmarks=["gsm8k", "arc_challenge", "commonsense_qa", "hotpot_qa", "svamp", "squad_v2"],
    samples_per_benchmark=2,
    streaming=True,
)

df_spectrum = pd.DataFrame([
    {
        "id": item["id"],
        "dataset": item["dataset"],
        "category": item["category"],
        "question": item["question"][:90] + "..." if len(item["question"]) > 90 else item["question"],
        "ground_truth": item["ground_truth"][:30],
    }
    for item in REAL_SPECTRUM
])

display(df_spectrum)

## 3. Live Trace Generation on Real Question (`rp.generate_trace`)

Prompts Generator $G$ live via `rp.generate_trace` on an authentic benchmark question. The resulting reasoning output is automatically parsed and segmented into discrete deduction steps.

In [ ]:
# Select a real problem from the spectrum (e.g. ARC-Challenge or GSM8K)
sample_problem = REAL_SPECTRUM[2]  # ARC-Challenge sample
question_text = sample_problem["question"]
category_name = sample_problem["category"]
dataset_name = sample_problem["dataset"]

print(f"📌 Source Dataset: {dataset_name} ({category_name})")
print(f"🎯 Question:\n{question_text}\n")
print(f"🔑 Ground Truth: {sample_problem['ground_truth']}")
print(f"\nGenerating live reasoning trace with Model G ({MODEL_G})...")

# Live generation call
live_trace = rp.generate_trace(
    question=question_text,
    model=MODEL_G,
    temperature=0.7,
)

print(f"\n✅ Generated {len(live_trace.steps)} reasoning steps ({live_trace.token_count} tokens):")
for i, step in enumerate(live_trace.steps):
    print(f"  [{i}] {step}")

## 4. Live Overthinking Audit (`rp.find_first_skip`)

Passes the live `ReasoningTrace` to Decision Auditor model $D$. The auditor determines whether intermediate steps contain conversational preambles, question restatements, redundant verification loops, or irrelevant detours that can be safely skipped.

In [ ]:
print(f"Auditing trace live with Decision Model ({MODEL_D})...")

# Live decision call
live_decision = rp.find_first_skip(
    trace=live_trace,
    decision_model=MODEL_D,
    temperature=0.0,
)

print("\n📋 Prune Decision Result:")
print(f"  Can Skip:        {live_decision.can_skip}")
if live_decision.can_skip:
    print(f"  Skip Span:       Steps [{live_decision.skip_start_idx} .. {live_decision.skip_end_idx}]")
    print(f"  Skipped Steps:   {live_decision.skipped_steps}")
    print(f"  Next Kept Step:  {live_decision.next_step}")
    print(f"  Auditor Reason:  {live_decision.reason}")
else:
    print(f"  Reason:          {live_decision.reason}")

## 5. Live Visual Trace Diff Rendering (`rp.render_trace_diff`)

Renders a transparent, color-coded HTML diff:
- 🟩 **Green text**: Kept context prefix ($x$)
- 🟥 **Red strikethrough**: Removable/redundant thoughts ($s_k$)
- 🟦 **Cyan text**: Next useful deduction target ($y$)

In [ ]:
# Render interactive HTML diff from live objects
html_diff = rp.render_trace_diff(live_trace, live_decision, as_html=True)
display(HTML(html_diff))

## 6. Live Transition Extraction (`rp.extract_transition`)

Transforms the audited trace into a training transition example $(x \to y)$ where the model learns to bypass the redundant span directly.

In [ ]:
if live_decision.can_skip:
    live_transition = rp.extract_transition(
        trace=live_trace,
        decision=live_decision,
        depth=1,
        example_id=f"{sample_problem['id']}_d1",
    )
    
    print("=" * 65)
    print(f"🎯 EXTRACTED PRUNING-TRANSITION PAIR: {live_transition.id}")
    print("=" * 65)
    print(f"📌 Input Context (x):\n{live_transition.input_x}\n")
    print(f"🚀 Target Continuation (y):\n{live_transition.target_y}\n")
    print(f"✂️ Skipped Thoughts:\n{live_transition.skipped_steps}\n")
    print(f"💡 Audit Justification:\n{live_transition.skip_reason}")
else:
    print("No skippable steps found for this trace; transition extraction not applicable.")

## 7. Live Multi-Depth Recursive Rollout on Multi-Step Math (`rp.rollout_pruning`)

Executes recursive multi-depth pruning across iterations on a real GSM8K / SVAMP arithmetic word problem. At each depth $d$, the pruned prefix is fed back into $G$, auditing the continuation for secondary redundancies until the final answer is reached.

In [ ]:
# Select real GSM8K math problem from spectrum
gsm8k_sample = next(item for item in REAL_SPECTRUM if "gsm8k" in item["dataset"])
rollout_q = gsm8k_sample["question"]

print(f"🔬 Running Live Multi-Depth Rollout on GSM8K Question:")
print(f"   {rollout_q}\n")

rollout_res = rp.rollout_pruning(
    question=rollout_q,
    generator_model=MODEL_G,
    decision_model=MODEL_D,
    max_depth=3,
)

print(f"\n✅ Rollout complete! Total Depths: {len(rollout_res.traces)}, Transitions Extracted: {len(rollout_res.transitions)}")
print(f"Original Steps: {rollout_res.original_step_count} ➔ Final Steps: {rollout_res.final_step_count} ({rollout_res.compression_ratio*100:.1f}% reduction)")

for i, tr in enumerate(rollout_res.transitions, 1):
    print(f"\n--- [Transition {i} | Depth {tr.depth}] ---")
    print(f"Input (x):  {tr.input_x[:80]}...")
    print(f"Target (y): {tr.target_y}")
    print(f"Skipped:    {tr.skipped_steps}")
    print(f"Reason:     {tr.skip_reason}")

## 8. Live Pruning-Transition Dataset Builder (`rp.build_pt_dataset`)

Converts the full multi-domain benchmark spectrum into a Hugging Face `Dataset` ready for 4-bit QLoRA SFT training.

In [ ]:
# Select representative real questions from each reasoning family
benchmark_questions = REAL_SPECTRUM[:6]

print(f"Building Hugging Face PT Dataset live across {len(benchmark_questions)} real benchmark questions...")
hf_dataset = rp.build_pt_dataset(
    questions=benchmark_questions,
    generator_model=MODEL_G,
    decision_model=MODEL_D,
    max_depth=2,
    max_workers=2,
)

print(f"\n✅ Built Live Dataset with {len(hf_dataset)} transition examples!")
if len(hf_dataset) > 0:
    df_ds = hf_dataset.to_pandas()
    display(df_ds[["id", "depth", "input_x", "target_y", "skip_reason"]].head(6))

# To synchronize the dataset directly to Hugging Face Hub:
# rp.push_dataset_to_hf(hf_dataset, "your-hf-username/rp-spectrum-v1")

## 9. Overthinking Audit across Reasoning Families (Cross-Domain Analysis)

Compare how overthinking patterns manifest differently across distinct reasoning domains (e.g. conversational preamble in word math vs. redundant property enumeration in science QA).

In [ ]:
print("🔬 Probing overthinking across distinct cognitive families:\n")
audit_summary = []

for item in REAL_SPECTRUM[:4]:
    trace = rp.generate_trace(question=item["question"], model=MODEL_G, temperature=0.7)
    decision = rp.find_first_skip(trace=trace, decision_model=MODEL_D, temperature=0.0)
    
    audit_summary.append({
        "Dataset": item["dataset"],
        "Category": item["category"],
        "Steps": len(trace.steps),
        "Can Skip": decision.can_skip,
        "Skipped Span": f"[{decision.skip_start_idx}..{decision.skip_end_idx}]" if decision.can_skip else "None",
        "Reason": decision.reason[:80] + "..." if len(decision.reason) > 80 else decision.reason,
    })

display(pd.DataFrame(audit_summary))